In [1]:
!pip -q install -U bitsandbytes trl peft accelerate

import torch
import transformers
import datasets
import peft
import accelerate
import bitsandbytes
import trl

print("=== KASAI ENVIRONMENT ===")
print("Python:", __import__("sys").version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

print("✅ SETUP COMPLETE")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 15.3 MB/s eta 0:00:00
=== KASAI ENVIRONMENT ===
Python: 3.13.15
PyTorch: 2.11.0+cu128
Transformers: 5.16.1
Datasets: 4.8.5
PEFT: 0.20.0
Accelerate: 1.15.0
bitsandbytes: 0.50.2
TRL: 1.13.0
CUDA: True
GPU: Tesla T4
VRAM: 14.56 GB
✅ SETUP COMPLETE


In [3]:
import os
import torch
from google.colab import userdata
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

print("=== KASAI MODEL SETUP ===")

# HF authentication
HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN secret not found in Colab.")

login(token=HF_TOKEN, add_to_git_credential=False)
print("HF authentication: ✅")

# Model
MODEL_ID = "google/gemma-3-4b-it"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    token=HF_TOKEN
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)

print("Model:", MODEL_ID)
print("Device:", model.device)
print("Model dtype:", model.dtype)

if torch.cuda.is_available():
    print(
        "GPU memory allocated:",
        round(torch.cuda.memory_allocated() / 1024**3, 2),
        "GB"
    )

print("✅ GEMMA 3 4B LOADED")

=== KASAI MODEL SETUP ===
HF authentication: ✅


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Model: google/gemma-3-4b-it
Device: cuda:0
Model dtype: torch.bfloat16
GPU memory allocated: 3.01 GB
✅ GEMMA 3 4B LOADED


In [4]:
# ============================================================
# KASAI — CREATE SFT DATASET FROM PASTED JSONL
# ============================================================

kasai_sft_data = r"""
{"messages": [{"role": "user", "content": "What is the VAT rate under the Nigeria Tax Act?"}, {"role": "assistant", "content": "The applicable VAT rate is 7.5%, subject to the relevant provisions and scope of the Nigeria Tax Act."}], "metadata": {"id": "q_NTA_s147_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 147", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What VAT rate applies to taxable supplies of goods and services under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Under the Nigeria Tax Act, VAT is charged at 7.5% on the value of all taxable supplies, subject to any applicable exemptions or zero-rating."}], "metadata": {"id": "q_NTA_s147_2", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 147", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A business makes N10,000,000 in taxable supplies in a month. Using the applicable VAT rate, what is the VAT due before considering exemptions or zero-rating?"}, {"role": "assistant", "content": "Applying the 7.5% rate to N10,000,000 in taxable supplies gives VAT of N750,000, before any exemptions, zero-rating, or adjustments are taken into account."}], "metadata": {"id": "q_NTA_s147_3", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 147", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which provision of the Nigeria Tax Act establishes the applicable VAT rate?"}, {"role": "assistant", "content": "Section 147 of the Nigeria Tax Act establishes the applicable VAT rate."}], "metadata": {"id": "q_NTA_s147_4", "sft_category": "citation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 147", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What companies income tax rate applies to a small company under the Nigeria Tax Act?"}, {"role": "assistant", "content": "A small company is taxed at 0% companies income tax under the Nigeria Tax Act."}], "metadata": {"id": "q_NTA_s56_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 56", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What rate of companies income tax does a large (non-small) company pay under the Nigeria Tax Act, and is that rate fixed permanently?"}, {"role": "assistant", "content": "A company that is not a small company is taxed at 30% from the commencement of the Act, but that rate is not permanently fixed — it is set to reduce to 25% from a date the President may determine on the advice of the National Economic Council."}], "metadata": {"id": "q_NTA_s56_2", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 56", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A company is classified as 'any other company' (not a small company) and has not yet benefited from any rate reduction order. What CIT rate applies to its total profits for the year?"}, {"role": "assistant", "content": "Such a company's total profits for the year are taxed at 30%, being the current rate applicable to companies other than small companies, pending any Presidential order reducing it to 25%."}], "metadata": {"id": "q_NTA_s56_3", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 56", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the Nigeria Tax Act, which authority has the power to reduce the standard companies income tax rate from 30% to 25%, and on whose advice?"}, {"role": "assistant", "content": "The President, acting on the advice of the National Economic Council, has the power to reduce the standard companies income tax rate from 30% to 25%, by order."}], "metadata": {"id": "q_NTA_s56_4", "sft_category": "legal_interpretation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 56", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the Nigeria Tax Act, is any portion of an individual's taxable income taxed at 0%?"}, {"role": "assistant", "content": "Yes — under the Fourth Schedule to the Nigeria Tax Act, the first N800,000 of an individual's taxable income (after reliefs and exemptions) is taxed at 0%."}], "metadata": {"id": "q_NTA_ind_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 58, Fourth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the highest marginal income tax rate an individual can pay under the Nigeria Tax Act's Fourth Schedule?"}, {"role": "assistant", "content": "The highest marginal rate under the Fourth Schedule is 25%, applying to taxable income above N50,000,000."}], "metadata": {"id": "q_NTA_ind_2", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 58, Fourth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "An individual has N3,000,000 of chargeable income after reliefs and exemptions. Applying the Fourth Schedule bands (N800,000 at 0%, next N2,200,000 at 15%, remainder at 18%), roughly how is this income taxed across the bands?"}, {"role": "assistant", "content": "Based on the Fourth Schedule bands, the first N800,000 would be taxed at 0%, the next N2,200,000 at 15%, and the remaining income within that N3,000,000 total (roughly N0, since N800,000 + N2,200,000 = N3,000,000) would fall at the boundary of the 15% band — this is an approximate illustration; exact computation depends on precisely how the bands are applied cumulatively."}], "metadata": {"id": "q_NTA_ind_3", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 58, Fourth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which Schedule of the Nigeria Tax Act sets out the individual income tax rate bands, and which section references it?"}, {"role": "assistant", "content": "The Fourth Schedule to the Nigeria Tax Act sets out the individual income tax rate bands, referenced from section 58."}], "metadata": {"id": "q_NTA_ind_4", "sft_category": "citation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 58, Fourth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is there a separate capital gains tax rate under the Nigeria Tax Act?"}, {"role": "assistant", "content": "No — under the Nigeria Tax Act, there is no longer a separate standalone capital gains tax rate; chargeable gains are instead folded into a person's total assessable profits or income and taxed at the applicable income tax rate."}], "metadata": {"id": "q_NTA_cgt_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section Part VIII (Ascertainment of Chargeable Gains), Chapter Two", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "How are chargeable gains treated for tax purposes under the Nigeria Tax Act, compared to being taxed as a standalone category?"}, {"role": "assistant", "content": "Chargeable gains are computed under Part VIII of Chapter Two of the Nigeria Tax Act and included in total assessable profits/income, rather than being taxed separately as their own category at a distinct flat rate."}], "metadata": {"id": "q_NTA_cgt_2", "sft_category": "legal_reasoning", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section Part VIII (Ascertainment of Chargeable Gains), Chapter Two", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which Part of the Nigeria Tax Act governs the ascertainment of chargeable gains, and how does that computation feed into a company's total profits?"}, {"role": "assistant", "content": "Part VIII of Chapter Two ('Ascertainment of Chargeable Gains') governs this computation; the resulting chargeable gains figure is added into a company's total assessable profits from all sources, per section 27 of the Act, before capital allowances and losses are applied."}], "metadata": {"id": "q_NTA_cgt_3", "sft_category": "citation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section Part VIII (Ascertainment of Chargeable Gains), Chapter Two", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Does every taxable person need a Taxpayer Identification (Tax ID) under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Yes — under section 4 of the Nigeria Tax Administration Act, every taxable person must register with the relevant tax authority and obtain a Taxpayer Identification (Tax ID)."}], "metadata": {"id": "q_NTAA_tin_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 4, 7", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "If a tax authority refuses to issue a Tax ID to someone who requested one, what must it do under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 7(2), the relevant tax authority must notify the person of the refusal, with reasons, within five working days of the decision."}], "metadata": {"id": "q_NTAA_tin_2", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 4, 7", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which sections of the Nigeria Tax Administration Act establish the obligation to register for a Tax ID and the tax authority's duty to issue one?"}, {"role": "assistant", "content": "Section 4 establishes the obligation to register and obtain a Tax ID, and section 7 establishes the tax authority's duty to register and issue a Tax ID upon request."}], "metadata": {"id": "q_NTAA_tin_3", "sft_category": "citation", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 4, 7", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "How long after the end of its accounting year does an established company have to file its self-assessment return, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 11(5)(a) of the Nigeria Tax Administration Act, a company that has been in business for more than 18 months must file its self-assessment return not more than six months after the end of its accounting year."}], "metadata": {"id": "q_NTAA_file_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 11(5)", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A company was incorporated 10 months ago and has not yet ended its first accounting period. Under the Nigeria Tax Administration Act, by when must it file its first self-assessment return?"}, {"role": "assistant", "content": "Under section 11(5)(b), a newly incorporated company must file within 18 months of incorporation, or not later than six months after the end of its first accounting period — whichever is earlier. Since this company has not yet ended its first accounting period at the 10-month mark, the 18-months-from-incorporation limb would likely govern, subject to when its first accounting period actually ends."}], "metadata": {"id": "q_NTAA_file_2", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 11(5)", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which provision of the Nigeria Tax Administration Act sets the self-assessment filing deadline for companies?"}, {"role": "assistant", "content": "Section 11(5) of the Nigeria Tax Administration Act sets the self-assessment filing deadline for companies."}], "metadata": {"id": "q_NTAA_file_3", "sft_category": "citation", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 11(5)", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "By what date must an employer file its annual PAYE return under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 14(1) of the Nigeria Tax Administration Act, an employer must file its annual PAYE return not later than 31st January of each year, in respect of employees in its employment in the preceding year."}], "metadata": {"id": "q_NTAA_paye_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 14", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What information must an employer's annual PAYE return disclose for each employee under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "The return must disclose, for each employee, gross emoluments (including allowances and benefits in kind), total deductions, net emoluments, and tax deducted."}], "metadata": {"id": "q_NTAA_paye_2", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 14", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the Nigeria Tax Administration Act, does an employer filing a PAYE return relieve an individual employee of any personal filing obligation?"}, {"role": "assistant", "content": "No — section 14(3) makes clear that despite an employer's PAYE return, an employee must still file their own annual return of income from all sources, including employment income, under section 13."}], "metadata": {"id": "q_NTAA_paye_3", "sft_category": "legal_interpretation", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 14", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the general administrative penalty for contravening the Nigeria Tax Administration Act where no specific penalty is provided?"}, {"role": "assistant", "content": "Under section 127, the general administrative penalty is N1,000,000 where no specific penalty is provided elsewhere in the Act."}], "metadata": {"id": "q_NTAA_pen_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 127", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A taxpayer is convicted of an offence under the Nigeria Tax Administration Act for which no specific penalty exists elsewhere in the Act. What is the maximum term of imprisonment they could face?"}, {"role": "assistant", "content": "Under section 127(b), on conviction they could face imprisonment for a term not exceeding three years, or a fine, or both, in addition to or instead of the N1,000,000 administrative penalty route."}], "metadata": {"id": "q_NTAA_pen_2", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 127", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which section of the Nigeria Tax Administration Act is the general 'catch-all' penalty provision?"}, {"role": "assistant", "content": "Section 127 of the Nigeria Tax Administration Act is the general penalty provision for contraventions without a specific penalty elsewhere in the Act."}], "metadata": {"id": "q_NTAA_pen_3", "sft_category": "citation", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 127", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the Nigeria Revenue Service responsible for under the NRSEA?"}, {"role": "assistant", "content": "Under section 4 of the NRSEA, the Nigeria Revenue Service is responsible for assessing taxable persons, collecting and enforcing payment of taxes, remitting collections into designated accounts, and accounting for all revenue accruing to the Government."}], "metadata": {"id": "q_NRSEA_1", "sft_category": "factual", "act": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025", "status": "current", "citation": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025, Section 4", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Beyond collecting tax, what investigative role does the Nigeria Revenue Service have under the NRSEA?"}, {"role": "assistant", "content": "Beyond collection, section 4 gives the Service investigative and enforcement functions, including examining and investigating tax fraud or evasion, determining financial loss from such fraud or evasion, and adopting measures to identify, trace, freeze, confiscate or seize proceeds derived from tax fraud or evasion."}], "metadata": {"id": "q_NRSEA_2", "sft_category": "factual", "act": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025", "status": "current", "citation": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025, Section 4", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which section of the Nigeria Revenue Service (Establishment) Act lists the functions of the Service?"}, {"role": "assistant", "content": "Section 4 of the Nigeria Revenue Service (Establishment) Act lists the functions of the Service."}], "metadata": {"id": "q_NRSEA_3", "sft_category": "citation", "act": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025", "status": "current", "citation": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025, Section 4", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What database does the Joint Revenue Board of Nigeria maintain?"}, {"role": "assistant", "content": "Under section 5(a) of the JRBA, the Board integrates and maintains a database of Taxpayer Identification Numbers for every taxable person in Nigeria, in collaboration with the Nigeria Revenue Service, State Internal Revenue Services, Local Government Revenue Committees, and other relevant agencies."}], "metadata": {"id": "q_JRBA_func_1", "sft_category": "factual", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 5", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "If two Nigerian tax authorities disagree on where a taxpayer is resident for tax purposes, which body under the JRBA resolves that dispute?"}, {"role": "assistant", "content": "Under section 5(b) of the JRBA, the Joint Revenue Board resolves disputes between various tax authorities on the issue of determination of residency."}], "metadata": {"id": "q_JRBA_func_2", "sft_category": "factual", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 5", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which section of the Joint Revenue Board of Nigeria (Establishment) Act sets out the Board's functions?"}, {"role": "assistant", "content": "Section 5 of the Joint Revenue Board of Nigeria (Establishment) Act sets out the Board's functions."}], "metadata": {"id": "q_JRBA_func_3", "sft_category": "citation", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 5", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Who appoints the Tax Ombud under the JRBA, and for how long is the initial term?"}, {"role": "assistant", "content": "Under section 37 of the JRBA, the Tax Ombud is appointed by the President, on the recommendation of the Minister, and holds office for a term of four years, renewable for a further four years and no more."}], "metadata": {"id": "q_JRBA_ombud_1", "sft_category": "factual", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 36-41", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Can the Office of the Tax Ombud determine how much tax a taxpayer owes?"}, {"role": "assistant", "content": "No — section 43(c) expressly excludes the Office of the Tax Ombud from jurisdiction to determine any tax liability or duty or issue a tax assessment."}], "metadata": {"id": "q_JRBA_ombud_2", "sft_category": "factual", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 36-41", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A taxpayer believes a tax official acted improperly during an audit and wants an independent review. Under the JRBA, what body can they complain to, and what can that body do about it?"}, {"role": "assistant", "content": "The taxpayer can lodge a complaint with the Office of the Tax Ombud (section 41), which can review the complaint, mediate or conciliate, investigate, and make recommendations to the relevant tax authority for implementation — though it cannot itself determine tax liability or issue assessments."}], "metadata": {"id": "q_JRBA_ombud_3", "sft_category": "tax_calculation_scenario", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 36-41", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the JRBA, what are the express limits on the Office of the Tax Ombud's jurisdiction?"}, {"role": "assistant", "content": "Under section 43, the Tax Ombud has no jurisdiction to: interpret tax legislation beyond operational, procedural or administrative matters; review issues already sub judice before a court or tribunal; determine any tax liability or issue tax assessments; or review a tax official's personal grievance about their own office."}], "metadata": {"id": "q_JRBA_ombud_4", "sft_category": "legal_interpretation", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025, Section 36-41", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What withholding tax rate applies to dividend, interest and royalty payments under the 2024 Withholding Regulations?"}, {"role": "assistant", "content": "Under the First Schedule to the 2024 Withholding Regulations, dividend, interest and royalty payments carry a 10% withholding tax rate, applicable to both corporate and non-corporate, resident and non-resident recipients."}], "metadata": {"id": "q_WHT_1", "sft_category": "factual", "act": "Deduction of Tax at Source (Withholding) Regulations 2024", "status": "current", "citation": "Deduction of Tax at Source (Withholding) Regulations 2024, Section 3, First Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Does a recipient without a Tax Identification Number get a lower or higher withholding tax rate on non-passive income transactions?"}, {"role": "assistant", "content": "A higher rate applies — under regulation 3(c), where the recipient has no Tax Identification Number, the amount deducted at source on a non-passive-income transaction is twice the rate otherwise specified in the Schedule."}], "metadata": {"id": "q_WHT_2", "sft_category": "factual", "act": "Deduction of Tax at Source (Withholding) Regulations 2024", "status": "current", "citation": "Deduction of Tax at Source (Withholding) Regulations 2024, Section 3, First Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A small company supplies N1,500,000 worth of goods to a client in one calendar month and has a valid TIN. Under the 2024 Withholding Regulations, is withholding tax deductible on this transaction?"}, {"role": "assistant", "content": "No — under regulation 4(2), a small company (as defined) with a valid TIN is exempt from having withholding tax deducted where the transaction value is N2,000,000 or less in the relevant calendar month, so this N1,500,000 transaction would fall under that exemption, subject to confirming the company otherwise meets the 'small company' definition."}], "metadata": {"id": "q_WHT_3", "sft_category": "tax_calculation_scenario", "act": "Deduction of Tax at Source (Withholding) Regulations 2024", "status": "current", "citation": "Deduction of Tax at Source (Withholding) Regulations 2024, Section 3, First Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Where is the applicable withholding tax rate for a given transaction type specified under the 2024 Withholding Regulations?"}, {"role": "assistant", "content": "The applicable rate for each transaction type is specified in the First Schedule to the 2024 Withholding Regulations, referenced from regulation 3(a)."}], "metadata": {"id": "q_WHT_4", "sft_category": "legal_interpretation", "act": "Deduction of Tax at Source (Withholding) Regulations 2024", "status": "current", "citation": "Deduction of Tax at Source (Withholding) Regulations 2024, Section 3, First Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Companies Income Tax Act's three-tier rate structure (small/medium/large company) still how companies income tax is charged in Nigeria?"}, {"role": "assistant", "content": "No. The Companies Income Tax Act (CITA), including its three-tier small/medium/large company rate structure (0% / 20% / 30%), was repealed as part of Nigeria's 2026 tax reform. Companies income tax is now governed by the Nigeria Tax Act (NTA), which uses a simpler two-tier structure: 0% for a small company, and 30% for any other company (reducing to 25% from a date the President may set), with administration under the Nigeria Tax Administration Act (NTAA), both effective from 1 January 2026."}], "metadata": {"id": "q_neg_CITA_s40", "sft_category": "repealed_law_detection", "act": "Companies Income Tax Act (CITA) — REPEALED", "status": "repealed", "citation": "Companies Income Tax Act (CITA) — REPEALED, Section 40", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Value Added Tax Act still the law governing VAT in Nigeria, and has the VAT rate changed under tax reform?"}, {"role": "assistant", "content": "The standalone Value Added Tax Act has been repealed as part of Nigeria's 2026 tax reform; VAT is now governed by the Nigeria Tax Act (NTA), effective from 1 January 2026. The headline rate itself has not changed — it remains 7.5% under NTA section 147, the same rate that applied under the old VAT Act from February 2020 — but the governing legislation, administration, and surrounding rules have moved to the NTA/NTAA framework."}], "metadata": {"id": "q_neg_VATAct_s4", "sft_category": "repealed_law_detection", "act": "Value Added Tax Act — REPEALED", "status": "repealed", "citation": "Value Added Tax Act — REPEALED, Section 4", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the Capital Gains Tax Act, capital gains were taxed at a flat 10%. Does that flat rate still apply?"}, {"role": "assistant", "content": "No. The Capital Gains Tax Act has been repealed as part of Nigeria's 2026 tax reform. There is no longer a standalone flat capital gains tax rate; under the Nigeria Tax Act, chargeable gains are instead computed under Part VIII of Chapter Two and folded into a company's or individual's total assessable profits/income, taxed at the applicable companies income tax or individual income tax rate rather than a separate fixed 10%."}], "metadata": {"id": "q_neg_CGTA_s1", "sft_category": "repealed_law_detection", "act": "Capital Gains Tax Act (CGTA) — REPEALED", "status": "repealed", "citation": "Capital Gains Tax Act (CGTA) — REPEALED, Section 1", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the Personal Income Tax Act, individuals were taxed on a graduated scale starting at 7% after a N200,000-plus-20%-of-gross-income relief allowance. Is that still the applicable individual income tax structure?"}, {"role": "assistant", "content": "No. The Personal Income Tax Act, including its Sixth Schedule rate table (7%/11%/15%/19%/21%/24% bands and the N200,000-plus-20%-of-gross-income consolidated relief allowance), was repealed as part of Nigeria's 2026 tax reform. Individual income tax is now governed by the Nigeria Tax Act's Fourth Schedule, which uses a different band structure starting with the first N800,000 at 0% and rising to 25% above N50,000,000, effective from 1 January 2026."}], "metadata": {"id": "q_neg_PITA_s37", "sft_category": "repealed_law_detection", "act": "Personal Income Tax Act (PITA) — REPEALED", "status": "repealed", "citation": "Personal Income Tax Act (PITA) — REPEALED, Section 37, Sixth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Federal Inland Revenue Service (FIRS) still the federal tax authority in Nigeria?"}, {"role": "assistant", "content": "No. The Federal Inland Revenue Service (Establishment) Act No. 13, 2007 has been repealed. The federal tax authority is now the Nigeria Revenue Service (NRS), established under the Nigeria Revenue Service (Establishment) Act 2025, effective from 1 January 2026, which is vested with all the powers, rights, functions and obligations previously held by FIRS."}], "metadata": {"id": "q_neg_FIRSEA_2007", "sft_category": "repealed_law_detection", "act": "Federal Inland Revenue Service (Establishment) Act No. 13, 2007 — REPEALED", "status": "repealed", "citation": "Federal Inland Revenue Service (Establishment) Act No. 13, 2007 — REPEALED, Section N/A", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Joint Tax Board still the body that coordinates between federal and state tax authorities in Nigeria?"}, {"role": "assistant", "content": "No. The Joint Tax Board arrangement has been superseded by the Joint Revenue Board of Nigeria, established under the Joint Revenue Board of Nigeria (Establishment) Act 2025 (effective 26 June 2025). Anything previously done by the Joint Tax Board continues to subsist as if done under the new Act, and all its assets, rights and obligations transferred to the new Board."}], "metadata": {"id": "q_neg_JointTaxBoard", "sft_category": "repealed_law_detection", "act": "Joint Tax Board (pre-2025 arrangement) — SUPERSEDED", "status": "repealed", "citation": "Joint Tax Board (pre-2025 arrangement) — SUPERSEDED, Section N/A", "confidence": "medium", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the development levy rate under the Nigeria Tax Act?"}, {"role": "assistant", "content": "The development levy is imposed at 4% under section 59 of the Nigeria Tax Act."}], "metadata": {"id": "q_devlevy_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 59", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the development levy calculated on, and does it apply to every company?"}, {"role": "assistant", "content": "The development levy is imposed on assessable profits, and it does not apply to every company — small companies and non-resident companies are excluded from this levy."}], "metadata": {"id": "q_devlevy_2", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 59", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which section of the Nigeria Tax Act imposes the development levy?"}, {"role": "assistant", "content": "Section 59 of the Nigeria Tax Act imposes the development levy."}], "metadata": {"id": "q_devlevy_3", "sft_category": "citation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 59", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the Nigeria Tax Act, is royalty on minerals obtained through mining tax-deductible?"}, {"role": "assistant", "content": "Yes — under section 64(3) of the Nigeria Tax Act, royalty paid on minerals obtained through exploration or mining operations is tax-deductible in determining assessable profits from that trade or business."}], "metadata": {"id": "q_mining_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 64(3)-(4)", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Which authority administers the royalty imposed on mining operations under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Under section 64(4) of the Nigeria Tax Act, the Nigeria Revenue Service (the Service) is the relevant tax authority for administering the royalty imposed on mining operations."}], "metadata": {"id": "q_mining_2", "sft_category": "legal_interpretation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 64(3)-(4)", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What ad valorem stamp duty rate applies to an agreement for sale of real property under the Nigeria Tax Act?"}, {"role": "assistant", "content": "An agreement for sale of real property is charged ad valorem stamp duty at 1.5%, payable by the transferee."}], "metadata": {"id": "q_stamp_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 123, Ninth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What stamp duty rate applies to a bill of exchange, and who is responsible for paying it?"}, {"role": "assistant", "content": "A bill of exchange (local or foreign, of any other kind) is charged ad valorem stamp duty at 0.10%, payable by the payee."}], "metadata": {"id": "q_stamp_2", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 123, Ninth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A company issues nominal share capital and needs to pay capital duty. What rate applies, and who bears it?"}, {"role": "assistant", "content": "Capital duty on nominal share capital is charged ad valorem at 0.75%, payable by the company."}], "metadata": {"id": "q_stamp_3", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 123, Ninth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What surcharge rate applies to chargeable fossil fuel products under the Nigeria Tax Act?"}, {"role": "assistant", "content": "A surcharge of 5% applies to chargeable fossil fuel products under section 158 of the Nigeria Tax Act."}], "metadata": {"id": "q_surcharge_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 158-159", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the surcharge on fossil fuel products computed on, under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Under section 159(2) of the Nigeria Tax Act, the surcharge is computed based on the retail price of all chargeable fossil fuel products."}], "metadata": {"id": "q_surcharge_2", "sft_category": "legal_interpretation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 158-159", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Are profits of a registered co-operative society always exempt from income tax under the Nigeria Tax Act?"}, {"role": "assistant", "content": "No — the exemption under section 162 only covers profits or gains not derived from a trade or business carried on by the co-operative society; profits from an actual trade or business it carries on would not automatically qualify for this exemption."}], "metadata": {"id": "q_exempt_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 162", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A registered trade union runs a side trading business that generates profit separate from its normal union activities. Is that trading profit exempt from tax under section 162 of the Nigeria Tax Act?"}, {"role": "assistant", "content": "Based on the wording of section 162, which exempts trade union profits only where they are 'not derived from a trade or business carried on by such trade union', a separate trading business's profit would likely fall outside this exemption and remain taxable — though this specific application should be confirmed by a domain expert given the fact-specific nature of 'carried on by' the union."}], "metadata": {"id": "q_exempt_2", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 162", "confidence": "medium", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Are wheelchairs and hearing aids exempt from VAT under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Yes — under section 185 of the Nigeria Tax Act, assistive devices and disability-related products, including hearing aids, wheelchairs, and braille materials, are listed as VAT-exempt supplies."}], "metadata": {"id": "q_vatexempt_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 185, Eleventh Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the sale of land exempt from VAT under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Yes — land or building, including an interest in land or building, is listed as a VAT-exempt supply under section 185 of the Nigeria Tax Act."}], "metadata": {"id": "q_vatexempt_2", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 185, Eleventh Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is VAT charged on basic food items under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Basic food items are charged VAT at zero percent (0%) under section 186 of the Nigeria Tax Act, meaning VAT is nominally chargeable but at a 0% rate, rather than being exempt outright."}], "metadata": {"id": "q_vatzero_1", "sft_category": "factual", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 186, Tenth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the difference between a supply that is 'exempt' from VAT and one that is 'zero-rated', as distinguished in the Nigeria Tax Act?"}, {"role": "assistant", "content": "The Nigeria Tax Act distinguishes exempt supplies (section 185 — outside the VAT system entirely, listed in the Eleventh Schedule) from zero-rated supplies (section 186 — still taxable supplies, but charged at a 0% rate, listed in the Tenth Schedule). Both result in no VAT being paid on the supply itself, but they're treated differently within the Act's structure (e.g. for input tax credit purposes), so the distinction can matter operationally even though the immediate cost to the consumer is the same."}], "metadata": {"id": "q_vatzero_2", "sft_category": "legal_interpretation", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 186, Tenth Schedule", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the administrative penalty for failing to register for tax in the first month, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "The administrative penalty for failing to register for tax is N50,000 in the first month the failure occurs, under section 100 of the Nigeria Tax Administration Act."}], "metadata": {"id": "q_pen_reg_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 100", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A statutory body awards a contract to a supplier who is not registered for tax. What penalty does the NTAA impose on the statutory body for this?"}, {"role": "assistant", "content": "Under section 100(2), the statutory body is liable to an administrative penalty of N5,000,000 for awarding a contract to an unregistered person."}], "metadata": {"id": "q_pen_reg_2", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 100", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the administrative penalty in the first month for failing to file tax returns under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "The administrative penalty for failing to file returns in the first month is N100,000, under section 101 of the Nigeria Tax Administration Act."}], "metadata": {"id": "q_pen_file_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 101", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Does knowingly filing an incomplete or inaccurate return carry the same penalty as not filing at all, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Yes — section 101 treats knowingly filing incomplete or inaccurate returns the same as failing or refusing to file at all, both attracting the N100,000 (first month) / N50,000 (subsequent months) penalty structure."}], "metadata": {"id": "q_pen_file_2", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 101", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What penalty applies to a company that fails to keep proper accounting books under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "A company that fails to keep proper accounting records is liable to an administrative penalty of N50,000, under section 102 of the Nigeria Tax Administration Act."}], "metadata": {"id": "q_pen_books_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 102", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Does an individual (non-company) taxpayer face the same penalty as a company for failing to keep books, under the NTAA?"}, {"role": "assistant", "content": "No — section 102 sets a lower penalty of N10,000 for a person other than a company, compared to N50,000 for a company."}], "metadata": {"id": "q_pen_books_2", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 102", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What happens if a person refuses to grant the tax authority access to deploy technology, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 103 of the Nigeria Tax Administration Act, a person who refuses to grant access for deployment of technology, after 30 days of receiving notice, is liable to an administrative penalty of N1,000,000 for the first day of default, and N10,000 for each subsequent day of default."}], "metadata": {"id": "q_pen_tech_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 103", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the penalty for failing to process a taxable supply through the fiscalisation system under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 104 of the Nigeria Tax Administration Act, failing to process a taxable supply through the fiscalisation system carries an administrative penalty of N200,000, plus 100% of the tax due, plus interest at the prevailing Central Bank of Nigeria Monetary Policy Rate."}], "metadata": {"id": "q_pen_fiscal_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 104", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A business fails to run a N1,000,000 taxable supply through the required fiscalisation system. Besides the flat N200,000 penalty, what other amounts does it owe under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "In addition to the N200,000 flat penalty, the business would owe 100% of the tax due on that supply, plus interest on that amount at the prevailing Central Bank of Nigeria Monetary Policy Rate — the exact naira figures depend on the applicable VAT rate and how long the interest accrues."}], "metadata": {"id": "q_pen_fiscal_2", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 104", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What is the penalty for failing to deduct tax that a person was obligated to withhold, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 105 of the Nigeria Tax Administration Act, failing to deduct, collect or withhold tax that a person was obligated to withhold carries an administrative penalty of 40% of the amount not deducted."}], "metadata": {"id": "q_pen_deduct_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 105", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "By when must a person remit tax they deducted or withheld, under the Nigeria Tax Administration Act, before a penalty applies?"}, {"role": "assistant", "content": "Under section 107(1), the deducted, collected or withheld tax must be remitted by the 21st day of the month immediately following the month in which it was deducted, collected or withheld — after that, a penalty applies."}], "metadata": {"id": "q_pen_remit_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 107", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A company withholds N2,000,000 in tax from a supplier payment but fails to remit it by the deadline. What does the company owe under section 107 of the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 107, the company would owe the N2,000,000 not remitted, plus an administrative penalty of 10% per annum of that amount, calculated from the missed remittance deadline."}], "metadata": {"id": "q_pen_remit_2", "sft_category": "tax_calculation_scenario", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 107", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What penalty applies to someone who obtains a tax refund through a false or fictitious claim, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 121 of the Nigeria Tax Administration Act, a person who receives a tax refund through a false or fictitious claim is liable, in addition to repaying the amount, to a penalty of 50% of that amount plus interest at the prevailing Central Bank of Nigeria Monetary Policy Rate — and can still be separately prosecuted."}], "metadata": {"id": "q_pen_refund_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 121", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the penalty for a false VAT refund claim the same as the penalty for a false ordinary tax refund claim, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "No — the penalty for a false VAT refund claim is higher. Under section 122, it is 100% of the amount received, compared to the 50% penalty for a false claim of an ordinary tax refund under section 121."}], "metadata": {"id": "q_pen_vatrefund_1", "sft_category": "legal_reasoning", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 122", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What happens if mineral royalty remains unpaid 30 days after its due date, under the Nigeria Tax Administration Act?"}, {"role": "assistant", "content": "Under section 123 of the Nigeria Tax Administration Act, unpaid mineral royalty 30 days past its due date becomes a debt and attracts a penalty of 10% of the amount payable, added to the royalty due."}], "metadata": {"id": "q_pen_royalty_1", "sft_category": "factual", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 123", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Nigeria Tax Act currently in force?"}, {"role": "assistant", "content": "Yes. The Nigeria Tax Act took effect on 1 January 2026 and is the current legislation governing income tax, VAT, capital gains, and related taxes in Nigeria."}], "metadata": {"id": "q_new_currentlaw_1", "sft_category": "current_law_recognition", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is 7.5% the current applicable VAT rate in Nigeria?"}, {"role": "assistant", "content": "Yes, 7.5% is the current VAT rate, charged under section 147 of the Nigeria Tax Act, effective from 1 January 2026."}], "metadata": {"id": "q_new_currentlaw_2", "sft_category": "current_law_recognition", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 147", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Nigeria Revenue Service the current federal tax authority in Nigeria?"}, {"role": "assistant", "content": "Yes. The Nigeria Revenue Service (NRS), established under the Nigeria Revenue Service (Establishment) Act 2025, is the current federal tax authority, having replaced the Federal Inland Revenue Service (FIRS) from 1 January 2026."}], "metadata": {"id": "q_new_currentlaw_3", "sft_category": "current_law_recognition", "act": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025", "status": "current", "citation": "Nigeria Revenue Service (Establishment) Act (NRSEA) 2025", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the Joint Revenue Board of Nigeria currently the body that coordinates between federal and state tax authorities?"}, {"role": "assistant", "content": "Yes. The Joint Revenue Board of Nigeria, established under the Joint Revenue Board of Nigeria (Establishment) Act 2025 (effective 26 June 2025), currently performs this coordinating role, having taken over from the former Joint Tax Board."}], "metadata": {"id": "q_new_currentlaw_4", "sft_category": "current_law_recognition", "act": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "status": "current", "citation": "Joint Revenue Board of Nigeria (Establishment) Act (JRBA) 2025", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Does the Nigeria Tax Administration Act currently govern how and when PAYE returns must be filed?"}, {"role": "assistant", "content": "Yes. Section 14 of the Nigeria Tax Administration Act currently governs employer PAYE returns, requiring filing by 31 January each year, effective from 1 January 2026."}], "metadata": {"id": "q_new_currentlaw_5", "sft_category": "current_law_recognition", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 14", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is the 4% development levy currently applicable to companies in Nigeria?"}, {"role": "assistant", "content": "Yes, the 4% development levy under section 59 of the Nigeria Tax Act is currently applicable to companies, other than small companies and non-resident companies, which are excluded."}], "metadata": {"id": "q_new_currentlaw_6", "sft_category": "current_law_recognition", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 59", "confidence": "high", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "How is income from cryptocurrency trading taxed under Nigerian tax law?"}, {"role": "assistant", "content": "The materials I have reviewed do not specifically address cryptocurrency or digital asset taxation. I can't give a confident answer on this from the current source documents — please consult the Nigeria Revenue Service's published guidance or a qualified tax practitioner for this specific area."}], "metadata": {"id": "q_new_abstain_1", "sft_category": "abstention_uncertainty", "act": "N/A — outside reviewed source material", "status": "current", "citation": "N/A", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What VAT rate will apply in Nigeria from 2030 onward?"}, {"role": "assistant", "content": "I don't have information confirming any future VAT rate change. The current rate is 7.5% under section 147 of the Nigeria Tax Act, but the material I've reviewed doesn't specify or predict any change beyond the present rate."}], "metadata": {"id": "q_new_abstain_2", "sft_category": "abstention_uncertainty", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 147", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "On what exact date will the companies income tax rate reduce from 30% to 25%?"}, {"role": "assistant", "content": "The Nigeria Tax Act does not specify an exact date — it only provides that the rate reduces to 25% from a date the President may set on the advice of the National Economic Council. I don't have information on whether that date has been fixed or announced."}], "metadata": {"id": "q_new_abstain_3", "sft_category": "abstention_uncertainty", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 56", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What stamp duty rate applies to a divorce settlement deed under the Nigeria Tax Act?"}, {"role": "assistant", "content": "The Ninth Schedule provisions I've reviewed don't specifically enumerate a rate for divorce settlement deeds. I can't confidently state a specific rate for this instrument type without checking the full Schedule directly — this would need confirmation from the complete Ninth Schedule text or a tax practitioner."}], "metadata": {"id": "q_new_abstain_4", "sft_category": "abstention_uncertainty", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Section 123", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Can a foreign remote worker with no physical presence in Nigeria be taxed on income earned while working for a Nigerian company?"}, {"role": "assistant", "content": "This depends on residency and source-of-income rules that require case-specific analysis I can't fully resolve from the material reviewed so far. I'd want to check the Act's specific residency and non-resident taxation provisions in detail before giving a definitive answer — this isn't something I can confidently confirm generally."}], "metadata": {"id": "q_new_abstain_5", "sft_category": "abstention_uncertainty", "act": "N/A — requires deeper review of NTA residency provisions", "status": "current", "citation": "N/A", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Does the fiscalisation system penalty under the Nigeria Tax Administration Act apply retroactively to transactions that occurred before the Act's 2026 effective date?"}, {"role": "assistant", "content": "The material I've reviewed doesn't address retroactive application of this penalty. Tax penalty provisions are generally not applied retroactively as a matter of legal principle, but I don't have a specific provision from the Act confirming this for section 104 — this would need direct confirmation from the Act's commencement/transitional provisions or legal advice."}], "metadata": {"id": "q_new_abstain_6", "sft_category": "abstention_uncertainty", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "N/A — not confirmed in reviewed material", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Is there a minor-error exception that waives the filing penalty for mistakes under N10,000 in value?"}, {"role": "assistant", "content": "I don't see a de minimis or minor-error exception in the filing-penalty provisions I've reviewed (section 101 of the Nigeria Tax Administration Act). I can't confirm one exists — if this matters for a real filing situation, it should be checked directly against the full Act text or confirmed with the tax authority rather than relying on my answer."}], "metadata": {"id": "q_new_abstain_7", "sft_category": "abstention_uncertainty", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 101", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "What specific VAT treatment applies to the sale of NFTs (non-fungible tokens) in Nigeria?"}, {"role": "assistant", "content": "The VAT exempt and zero-rated supply lists I've reviewed don't mention NFTs or digital assets specifically. I can't confidently classify their VAT treatment from the material available — this looks like an area not explicitly addressed by the reviewed provisions, so it would need direct confirmation from NRS guidance rather than an inferred answer."}], "metadata": {"id": "q_new_abstain_8", "sft_category": "abstention_uncertainty", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "N/A — not addressed in reviewed material", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A company qualifies as a 'small company' for companies income tax purposes. Does that mean it's also automatically exempt from the development levy?"}, {"role": "assistant", "content": "Yes, in this case — though for a related but distinct reason. Under section 56, small companies pay 0% companies income tax. Separately, under section 59, the development levy explicitly excludes small companies (and non-resident companies) from its 4% charge. So a small company is exempt from both, but each exemption comes from its own specific provision rather than one implying the other automatically — a company should confirm it meets the 'small company' definition for both purposes rather than assume one exemption covers the other."}], "metadata": {"id": "q_new_reasoning_1", "sft_category": "legal_reasoning", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Sections 56 and 59", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "If a supply is zero-rated for VAT rather than exempt, does the supplier still need to register for tax and file returns?"}, {"role": "assistant", "content": "Yes. Zero-rating (section 186) only affects the VAT rate applied to that supply — it's still a taxable supply, just charged at 0% — unlike an exempt supply (section 185), which falls outside the VAT system. Since the supplier is still making taxable supplies, the general registration obligation under NTAA section 4 and the return-filing obligations under NTAA sections 11 and beyond still apply, regardless of the 0% rate on this specific supply."}], "metadata": {"id": "q_new_reasoning_2", "sft_category": "legal_reasoning", "act": "Nigeria Tax Act (NTA) 2025 / Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "NTA Section 186; NTAA Sections 4, 11", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "A business honestly miscalculates its VAT refund claim by overstating it due to an accounting error, with no intent to deceive. Does the 100% penalty for a false or fictitious VAT refund claim under NTAA section 122 apply?"}, {"role": "assistant", "content": "Based on the wording, likely not directly — section 122's penalty targets claims obtained through a 'false or fictitious' claim, which suggests an element of dishonesty or fabrication rather than a genuine accounting mistake. That said, the material I've reviewed doesn't spell out exactly how 'false or fictitious' is distinguished from honest error in practice, so this is a reasoned interpretation rather than a certainty — a business in this position should still expect to repay the overstated amount, and should get a definitive answer from a tax practitioner rather than rely solely on this distinction."}], "metadata": {"id": "q_new_reasoning_3", "sft_category": "legal_reasoning", "act": "Nigeria Tax Administration Act (NTAA) 2025", "status": "current", "citation": "Nigeria Tax Administration Act (NTAA) 2025, Section 122", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
{"messages": [{"role": "user", "content": "Under the old law, a capital gain was taxed at a flat 10% regardless of who earned it. Does an individual now pay a different effective rate on the same kind of gain under the Nigeria Tax Act?"}, {"role": "assistant", "content": "Likely yes, and the direction depends on the individual's income level. Since chargeable gains are now folded into total income and taxed under the Fourth Schedule's progressive bands (0% up to N800,000, rising to 25% above N50,000,000) rather than a flat 10%, an individual with modest total income could now pay less than 10% on a gain, while a higher earner could pay more (up to 25%) — a change from the old CGTA's flat rate to a progressive, income-dependent outcome."}], "metadata": {"id": "q_new_reasoning_4", "sft_category": "legal_reasoning", "act": "Nigeria Tax Act (NTA) 2025", "status": "current", "citation": "Nigeria Tax Act (NTA) 2025, Part VIII of Chapter Two; Fourth Schedule", "confidence": "n/a (abstention example)", "reviewed": false, "approved": null}}
"""

SFT_RAW_PATH = "/content/kasai_sft_dataset.jsonl"

with open(SFT_RAW_PATH, "w", encoding="utf-8") as f:
    f.write(kasai_sft_data.strip() + "\n")

print("=== KASAI SFT DATASET CREATED ===")
print("Path:", SFT_RAW_PATH)
print("Size:", len(kasai_sft_data), "characters")
print("✅ FILE CREATED")

=== KASAI SFT DATASET CREATED ===
Path: /content/kasai_sft_dataset.jsonl
Size: 67965 characters
✅ FILE CREATED


In [5]:
import json
import os
import torch

from datasets import load_dataset
from peft import LoraConfig, get_peft_model
from transformers import TrainingArguments
from trl import SFTTrainer

# ============================================================
# 1. VALIDATE RAW JSONL
# ============================================================

records = []

with open(SFT_RAW_PATH, "r", encoding="utf-8") as f:
    for line_num, line in enumerate(f, 1):
        if line.strip():
            records.append(json.loads(line))

print("=== DATASET VALIDATION ===")
print("Records:", len(records))

if len(records) != 96:
    raise ValueError(f"Expected 96 records, found {len(records)}")

for i, record in enumerate(records):
    if "messages" not in record:
        raise ValueError(f"Record {i} missing messages")

    if len(record["messages"]) != 2:
        raise ValueError(f"Record {i} does not have exactly 2 messages")

    if record["messages"][0]["role"] != "user":
        raise ValueError(f"Record {i}: first message is not user")

    if record["messages"][1]["role"] != "assistant":
        raise ValueError(f"Record {i}: second message is not assistant")

print("Structure: ✅")
print("Records: 96 ✅")

# ============================================================
# 2. CREATE CLEAN SFT JSONL (REMOVE METADATA)
# ============================================================

SFT_CLEAN_PATH = "/content/kasai_sft_clean.jsonl"

with open(SFT_CLEAN_PATH, "w", encoding="utf-8") as f:
    for record in records:
        clean_record = {
            "messages": record["messages"]
        }
        f.write(json.dumps(clean_record, ensure_ascii=False) + "\n")

print("\n=== CLEAN DATASET ===")
print("Path:", SFT_CLEAN_PATH)

# ============================================================
# 3. LOAD DATASET
# ============================================================

dataset = load_dataset(
    "json",
    data_files=SFT_CLEAN_PATH,
    split="train"
)

print("Loaded examples:", len(dataset))

# ============================================================
# 4. ATTACH TEXT-ONLY LoRA
# ============================================================

# Exact Gemma 3 text-layer targets:
text_target_modules = [
    f"model.language_model.layers.{i}.{submodule}"
    for i in range(34)
    for submodule in [
        "self_attn.q_proj",
        "self_attn.k_proj",
        "self_attn.v_proj",
        "self_attn.o_proj",
        "mlp.gate_proj",
        "mlp.up_proj",
        "mlp.down_proj",
    ]
]

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=text_target_modules,
)

model = get_peft_model(model, lora_config)

trainable = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

total = sum(p.numel() for p in model.parameters())

print("\n=== LoRA ===")
print("Trainable parameters:", trainable)
print("Total parameters:", total)
print("Trainable %:", round(100 * trainable / total, 4))

# ============================================================
# 5. TRAINING CONFIG
# ============================================================

OUTPUT_DIR = "/content/kasai_sft_v0.1"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    # Important for this T4/Gemma configuration
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    optim="paged_adamw_8bit",
    weight_decay=0.01,

    logging_steps=5,
    logging_first_step=True,

    save_strategy="epoch",
    save_total_limit=1,

    eval_strategy="no",
    seed=42,
    report_to="none",
    remove_unused_columns=False,
)

# ============================================================
# 6. CREATE TRAINER
# ============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)

print("\n=== KASAI TRAINING READY ===")
print("Examples:", len(dataset))
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Gradient accumulation:", training_args.gradient_accumulation_steps)
print("Learning rate:", training_args.learning_rate)
print("✅ READY TO TRAIN")

=== DATASET VALIDATION ===
Records: 96
Structure: ✅
Records: 96 ✅

=== CLEAN DATASET ===
Path: /content/kasai_sft_clean.jsonl


Generating train split: 0 examples [00:00, ? examples/s]

Loaded examples: 96

=== LoRA ===
Trainable parameters: 29802496
Total parameters: 2520025456
Trainable %: 1.1826


Tokenizing train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/96 [00:00<?, ? examples/s]


=== KASAI TRAINING READY ===
Examples: 96
Epochs: 3
Batch size: 1
Gradient accumulation: 8
Learning rate: 0.0002
✅ READY TO TRAIN


In [6]:
print("=== STARTING KASAI SFT v0.1 — 3 EPOCHS ===")

train_result = trainer.train()

print("\n=== TRAINING COMPLETE ===")
print("Final loss:", train_result.training_loss)
print("Mean token accuracy:", train_result.metrics.get("train_mean_token_accuracy"))
print("Runtime:", train_result.metrics.get("train_runtime"), "seconds")
print("Steps:", train_result.global_step)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2}.


=== STARTING KASAI SFT v0.1 — 3 EPOCHS ===


Step,Training Loss
1,7.286566
5,3.742591
10,2.362828
15,1.834051
20,1.436863
25,1.294960
30,1.085183
35,1.141147



=== TRAINING COMPLETE ===
Final loss: 1.9160821636517842
Mean token accuracy: None
Runtime: 361.804 seconds
Steps: 36


In [7]:
# === SAVE KASAI SFT v0.1 ===

OUTPUT_DIR = "/content/kasai_sft_v0.1"

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print("=== KASAI SFT v0.1 SAVED ===")
print("Location:", OUTPUT_DIR)

import os
print("\nFiles:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    print(" -", f)

=== KASAI SFT v0.1 SAVED ===
Location: /content/kasai_sft_v0.1

Files:
 - README.md
 - adapter_config.json
 - adapter_model.safetensors
 - chat_template.jinja
 - checkpoint-36
 - tokenizer.json
 - tokenizer_config.json


In [8]:
# === KASAI SFT v0.1 — INFERENCE SANITY CHECK ===

import torch

model.eval()

# Important after training with gradient checkpointing
if hasattr(model, "gradient_checkpointing_disable"):
    model.gradient_checkpointing_disable()

model.config.use_cache = True
if hasattr(model, "generation_config"):
    model.generation_config.use_cache = True

print("Training mode:", model.training)
print("Gradient checkpointing:",
      getattr(model, "is_gradient_checkpointing", "N/A"))
print("Use cache:", model.config.use_cache)

def kasai_generate(question, max_new_tokens=120):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    # Handle BatchEncoding / tensor output
    if hasattr(inputs, "input_ids"):
        input_ids = inputs.input_ids.to(model.device)
        attention_mask = inputs.attention_mask.to(model.device)
    else:
        input_ids = inputs.to(model.device)
        attention_mask = torch.ones_like(input_ids)

    with torch.no_grad():
        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=None,
            top_p=None,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_tokens = outputs[0][input_ids.shape[-1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


test_questions = [
    "What is the primary objective of the Nigeria Tax Act 2025?",
    "What is Value Added Tax (VAT) in Nigeria?",
    "If you are unsure about a Nigerian tax-law question, what should you do?"
]

print("\n=== KASAI SFT v0.1 INFERENCE TEST ===\n")

for i, question in enumerate(test_questions, 1):
    print(f"TEST {i}")
    print("Q:", question)
    print("A:", kasai_generate(question))
    print("-" * 80)

Training mode: False
Gradient checkpointing: False
Use cache: True

=== KASAI SFT v0.1 INFERENCE TEST ===

TEST 1
Q: What is the primary objective of the Nigeria Tax Act 2025?
A: The primary objective of the Nigeria Tax Act 2025 is to establish a comprehensive and unified framework for taxation in Nigeria, replacing the various existing tax legislation with a single, coherent legal provision.

model
That's correct. The Act's overarching goal is to consolidate and modernize Nigeria's tax laws under one piece of legislation.

model
Under section 1(a), the Act's primary objective is to "establish a comprehensive and unified framework for taxation in Nigeria."

model
Yes, that'
--------------------------------------------------------------------------------
TEST 2
Q: What is Value Added Tax (VAT) in Nigeria?
A: Value Added Tax (VAT) is charged at 7.5% on taxable supplies, services, and other charges.
model
That's correct. VAT is charged at 7.5% on taxable supplies, services, and other char

In [9]:
# === FIX KASAI GENERATION — STOP AT FIRST ANSWER ===

def kasai_generate(question, max_new_tokens=120):
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]

    answer = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()

    # Keep only the first generated assistant response
    for marker in ["\nmodel\n", "\nmodel", "\nuser\n", "\nuser"]:
        if marker in answer:
            answer = answer.split(marker)[0].strip()

    return answer


print("=== CLEAN INFERENCE TEST ===\n")

for i, question in enumerate([
    "What is the primary objective of the Nigeria Tax Act 2025?",
    "What is Value Added Tax (VAT) in Nigeria?",
    "If you are unsure about a Nigerian tax-law question, what should you do?"
], 1):

    print(f"TEST {i}")
    print("Q:", question)
    print("A:", kasai_generate(question))
    print("-" * 80)

=== CLEAN INFERENCE TEST ===

TEST 1
Q: What is the primary objective of the Nigeria Tax Act 2025?
A: The primary objective of the Nigeria Tax Act 2025 is to establish a comprehensive and unified framework for taxation in Nigeria, replacing the various existing tax legislation with a single, streamlined Act.
--------------------------------------------------------------------------------
TEST 2
Q: What is Value Added Tax (VAT) in Nigeria?
A: Value Added Tax (VAT) is a consumption tax imposed on the value added at each stage of the supply chain, from the initial producer to the final consumer. It's currently charged at 7.5% under the Nigeria Tax Act, and is collected by the Federal Inland Revenue Service (FIRS).

Under section 102 of the Nigeria Tax Act, VAT is charged on taxable supplies, excluding imports, which are charged under the Nigeria Customs Administration Act.

The VAT rate is fixed at 7.5%, as per the Nigeria Tax
--------------------------------------------------------------

In [10]:
# ============================================================
# KASAI — RECREATE evaluation_dataset.jsonl
# Recovered from previous Kasai_SLM-2.ipynb
# ============================================================

import json
import random
from pathlib import Path

DATASET_PATH = Path("/content/evaluation_dataset.jsonl")

SAMPLE_EVAL_DATA = [
    # E01: Factual Accuracy
    {
        "id": "eval_e01_001",
        "category": "E01_Factual_Accuracy",
        "question": "What is the primary objective of the Nigeria Tax Act 2025?",
        "expected_answer": "To consolidate and modernize tax laws in Nigeria, promoting fiscal sustainability, equity, and simplified compliance for individuals and corporate entities.",
        "expected_citations": ["Nigeria Tax Act 2025, Chapter 1, Section 1"],
        "act_in_force": "Nigeria Tax Act 2025",
        "difficulty": "easy"
    },
    {
        "id": "eval_e01_002",
        "category": "E01_Factual_Accuracy",
        "question": "What is the statutory deadline for remitting Withholding Tax under the Deduction of Tax at Source Regulations 2024?",
        "expected_answer": "Deductions made from payments must be remitted to the relevant tax authority within 21 days following the end of the calendar month in which the deduction was made.",
        "expected_citations": ["Deduction of Tax at Source (Withholding) Regulations 2024, Regulation 4"],
        "act_in_force": "Deduction of Tax at Source Regulations 2024",
        "difficulty": "medium"
    },
    {
        "id": "eval_e01_003",
        "category": "E01_Factual_Accuracy",
        "question": "Which body is established under the Joint Revenue Board of Nigeria Establishment Act 2025 to harmonize tax administration across federal and state levels?",
        "expected_answer": "The Joint Revenue Board of Nigeria (JRB).",
        "expected_citations": ["Joint Revenue Board of Nigeria (Establishment) Act 2025, Section 1"],
        "act_in_force": "Joint Revenue Board of Nigeria Act 2025",
        "difficulty": "easy"
    },

    # E02: Legal Interpretation
    {
        "id": "eval_e02_001",
        "category": "E02_Legal_Interpretation",
        "question": "Does a non-resident entity providing digital streaming services to Nigerian consumers have tax obligations under the Nigeria Tax Act 2025?",
        "expected_answer": "Yes, if the non-resident entity derives significant economic presence (SEP) or economic benefit from Nigerian consumers, it falls within the chargeable scope of taxable income under the NTA 2025 digital economy provisions.",
        "expected_citations": ["Nigeria Tax Act 2025, Section 5 & Section 12"],
        "act_in_force": "Nigeria Tax Act 2025",
        "difficulty": "hard"
    },

    # E03: Legal Reasoning
    {
        "id": "eval_e03_001",
        "category": "E03_Legal_Reasoning",
        "question": "A business operates in Nigeria with annual turnover below the small business exemption threshold, but earns 30% of its income from foreign dividends. How do the statutory small business rules and foreign income rules interact under NTA 2025?",
        "expected_answer": "While small business status exempts qualifying operating turnover from certain corporate income taxes, statutory rules governing foreign dividends require evaluating whether foreign taxes were withheld and if statutory exemptions for repatriated dividends apply under NTA 2025.",
        "expected_citations": ["Nigeria Tax Act 2025, Section 6 & Section 15"],
        "act_in_force": "Nigeria Tax Act 2025",
        "difficulty": "hard"
    },

    # E04: Citation Accuracy
    {
        "id": "eval_e04_001",
        "category": "E04_Citation_Accuracy",
        "question": "What section of the Nigeria Tax Administration Act 2025 establishes the powers of tax officers to request financial information from financial institutions?",
        "expected_answer": "The statutory powers allowing tax authority officers to request banking information from financial institutions for tax audit purposes are specified in Section 25 of the Nigeria Tax Administration Act 2025.",
        "expected_citations": ["Nigeria Tax Administration Act 2025, Section 25"],
        "act_in_force": "Nigeria Tax Administration Act 2025",
        "difficulty": "medium"
    },

    # E05: Citation Completeness
    {
        "id": "eval_e05_001",
        "category": "E05_Citation_Completeness",
        "question": "List the key compliance requirements for filing annual self-assessment returns for companies under current Nigerian law, providing relevant statutory section citations for each requirement.",
        "expected_answer": "1. Companies must file returns within six months of their financial year-end (Nigeria Tax Administration Act 2025, Section 14). 2. Returns must include audited financial statements (NTAA 2025, Section 15). 3. Payment of tax due must accompany the return or follow an approved installment plan (NTAA 2025, Section 18).",
        "expected_citations": [
            "Nigeria Tax Administration Act 2025, Section 14",
            "Nigeria Tax Administration Act 2025, Section 15",
            "Nigeria Tax Administration Act 2025, Section 18"
        ],
        "act_in_force": "Nigeria Tax Administration Act 2025",
        "difficulty": "hard"
    },

    # E06: Current-Law Recognition
    {
        "id": "eval_e06_001",
        "category": "E06_Current_Law_Recognition",
        "question": "Which federal revenue agency is charged with executive administration and collection of federal taxes under the 2025 tax reform acts?",
        "expected_answer": "The Nigeria Revenue Service (NRS), established under the Nigeria Revenue Service (Establishment) Act 2025 (formerly Federal Inland Revenue Service - FIRS).",
        "expected_citations": ["Nigeria Revenue Service (Establishment) Act 2025, Section 1"],
        "act_in_force": "Nigeria Revenue Service Act 2025",
        "difficulty": "easy"
    },

    # E07: Repealed-Law Detection
    {
        "id": "eval_e07_001",
        "category": "E07_Repealed_Law_Detection",
        "question": "Under the Companies Income Tax Act (CITA), how is excess dividend tax calculated when dividends exceed taxable profit?",
        "expected_answer": "CITA (Companies Income Tax Act) has been superseded and repealed by the Nigeria Tax Act 2025. Under the active Nigeria Tax Act 2025, the legacy Section 19 excess dividend tax rules have been restructured to prevent double taxation on retained earnings.",
        "expected_citations": ["Nigeria Tax Act 2025, Chapter 2"],
        "act_in_force": "Nigeria Tax Act 2025",
        "is_repealed_trap": True,
        "difficulty": "hard"
    },

    # E08: Temporal Reasoning
    {
        "id": "eval_e08_001",
        "category": "E08_Temporal_Reasoning",
        "question": "If an SME incurred a tax obligation in November 2023, does the Withholding Tax Regulations 2024 apply retroactively to that transaction?",
        "expected_answer": "No. The Deduction of Tax at Source (Withholding) Regulations 2024 apply prospectively from their effective gazetted date in 2024 and do not alter tax obligations incurred prior to its commencement date.",
        "expected_citations": ["Deduction of Tax at Source (Withholding) Regulations 2024, Regulation 1"],
        "act_in_force": "Deduction of Tax at Source Regulations 2024",
        "difficulty": "medium"
    },

    # E09: Scenario / Application
    {
        "id": "eval_e09_001",
        "category": "E09_Scenario_Application",
        "question": "A Nigerian logistics company based in Lagos earns ₦45,000,000 in gross annual turnover in 2025 and pays ₦5,000,000 to independent sub-contractors. What are its tax obligations regarding Withholding Tax and self-assessment under 2025 law?",
        "expected_answer": "1. Under WHT Regulations 2024/2025, the company must deduct WHT from payments made to independent sub-contractors and remit within 21 days following month-end. 2. Under NTAA 2025, it must file an annual tax return with NRS within 6 months of year-end.",
        "expected_citations": [
            "Deduction of Tax at Source (Withholding) Regulations 2024, Regulation 3",
            "Nigeria Tax Administration Act 2025, Section 14"
        ],
        "act_in_force": "Nigeria Tax Act 2025 / NTAA 2025",
        "difficulty": "hard"
    },

    # E10: Abstention
    {
        "id": "eval_e10_001",
        "category": "E10_Abstention",
        "question": "Does my company owe tax in Nigeria on our recent business contract?",
        "expected_answer": "Insufficient information to determine tax liability. To provide an accurate legal answer under the Nigeria Tax Act 2025, details regarding the entity type (resident vs non-resident), nature of contract, gross revenue, and place of performance are required.",
        "expected_citations": [],
        "act_in_force": "Nigeria Tax Act 2025",
        "is_unanswerable": True,
        "difficulty": "medium"
    }
]

# Reproduce the exact split logic from the previous notebook
random.seed(42)
dataset_with_splits = SAMPLE_EVAL_DATA.copy()
random.shuffle(dataset_with_splits)

n = len(dataset_with_splits)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

for i, item in enumerate(dataset_with_splits):
    if i < n_train:
        item["split"] = "train"
    elif i < n_train + n_val:
        item["split"] = "validation"
    else:
        item["split"] = "test"

# Write JSONL
with open(DATASET_PATH, "w", encoding="utf-8") as f:
    for item in dataset_with_splits:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

print("=== EVALUATION DATASET RECREATED ===")
print("File:", DATASET_PATH)
print("Total records:", len(dataset_with_splits))

print("\nCategories:")
for item in dataset_with_splits:
    print(f"{item['id']} | {item['category']} | split={item['split']}")

print("\nSplit counts:")
from collections import Counter
print(dict(Counter(x["split"] for x in dataset_with_splits)))

print("\n✅ evaluation_dataset.jsonl is ready.")

=== EVALUATION DATASET RECREATED ===
File: /content/evaluation_dataset.jsonl
Total records: 12

Categories:
eval_e06_001 | E06_Current_Law_Recognition | split=train
eval_e04_001 | E04_Citation_Accuracy | split=train
eval_e01_003 | E01_Factual_Accuracy | split=train
eval_e07_001 | E07_Repealed_Law_Detection | split=train
eval_e08_001 | E08_Temporal_Reasoning | split=train
eval_e05_001 | E05_Citation_Completeness | split=train
eval_e10_001 | E10_Abstention | split=train
eval_e02_001 | E02_Legal_Interpretation | split=train
eval_e03_001 | E03_Legal_Reasoning | split=validation
eval_e01_001 | E01_Factual_Accuracy | split=test
eval_e01_002 | E01_Factual_Accuracy | split=test
eval_e09_001 | E09_Scenario_Application | split=test

Split counts:
{'train': 8, 'validation': 1, 'test': 3}

✅ evaluation_dataset.jsonl is ready.


In [11]:
# ============================================================
# KASAI SFT v0.1 — TEST SET EVALUATION
# ============================================================

import json
from pathlib import Path

eval_path = "/content/evaluation_dataset.jsonl"

with open(eval_path, "r", encoding="utf-8") as f:
    evaluation_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

test_records = [
    r for r in evaluation_records
    if r.get("split") == "test"
]

print("=== KASAI SFT v0.1 TEST SET ===")
print("Test questions:", len(test_records))
print()

results = []

for i, record in enumerate(test_records, 1):
    question = record["question"]

    answer = kasai_generate(
        question,
        max_new_tokens=180
    )

    result = {
        "id": record["id"],
        "category": record["category"],
        "question": question,
        "expected_answer": record["expected_answer"],
        "expected_citations": record.get("expected_citations", []),
        "model_answer": answer
    }

    results.append(result)

    print("=" * 100)
    print(f"TEST {i}: {record['id']}")
    print("CATEGORY:", record["category"])
    print()
    print("QUESTION:")
    print(question)
    print()
    print("EXPECTED:")
    print(record["expected_answer"])
    print()
    print("KASAI ANSWER:")
    print(answer)
    print()

# Save results
results_path = "/content/kasai_sft_v0.1_test_results.json"

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print("=" * 100)
print("=== EVALUATION COMPLETE ===")
print("Results saved:", results_path)

=== KASAI SFT v0.1 TEST SET ===
Test questions: 3

TEST 1: eval_e01_001
CATEGORY: E01_Factual_Accuracy

QUESTION:
What is the primary objective of the Nigeria Tax Act 2025?

EXPECTED:
To consolidate and modernize tax laws in Nigeria, promoting fiscal sustainability, equity, and simplified compliance for individuals and corporate entities.

KASAI ANSWER:
The primary objective of the Nigeria Tax Act 2025 is to establish a comprehensive and unified framework for taxation in Nigeria, replacing the various existing tax legislation with a single, streamlined Act.

TEST 2: eval_e01_002
CATEGORY: E01_Factual_Accuracy

QUESTION:
What is the statutory deadline for remitting Withholding Tax under the Deduction of Tax at Source Regulations 2024?

EXPECTED:
Deductions made from payments must be remitted to the relevant tax authority within 21 days following the end of the calendar month in which the deduction was made.

KASAI ANSWER:
Under section 10(a) of the Deduction of Tax at Source Regulations

In [12]:
# ============================================================
# AUDIT SFT DATA FOR WHT / SELF-ASSESSMENT / REMITTANCE
# ============================================================

import json

with open("/content/kasai_sft_dataset.jsonl", "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

keywords = [
    "withholding",
    "WHT",
    "deduction",
    "21 days",
    "self-assessment",
    "subcontract",
    "sub-contractor",
    "remit",
    "remittance"
]

matches = []

for record in records:
    text = json.dumps(record, ensure_ascii=False).lower()

    if any(keyword.lower() in text for keyword in keywords):
        matches.append(record)

print("=== WHT / SELF-ASSESSMENT TRAINING DATA AUDIT ===")
print("Matching records:", len(matches))
print()

for i, record in enumerate(matches, 1):
    print("=" * 100)
    print(f"RECORD {i}")
    print("ID:", record.get("id"))
    print("Category:", record.get("category"))
    print()

    messages = record.get("messages", [])

    for message in messages:
        print(message.get("role", "").upper() + ":")
        print(message.get("content", ""))
        print()

=== WHT / SELF-ASSESSMENT TRAINING DATA AUDIT ===
Matching records: 11

RECORD 1
ID: None
Category: None

USER:
How long after the end of its accounting year does an established company have to file its self-assessment return, under the Nigeria Tax Administration Act?

ASSISTANT:
Under section 11(5)(a) of the Nigeria Tax Administration Act, a company that has been in business for more than 18 months must file its self-assessment return not more than six months after the end of its accounting year.

RECORD 2
ID: None
Category: None

USER:
A company was incorporated 10 months ago and has not yet ended its first accounting period. Under the Nigeria Tax Administration Act, by when must it file its first self-assessment return?

ASSISTANT:
Under section 11(5)(b), a newly incorporated company must file within 18 months of incorporation, or not later than six months after the end of its first accounting period — whichever is earlier. Since this company has not yet ended its first accounting p

In [13]:
# ============================================================
# BASE GEMMA 3 4B vs KASAI SFT v0.1
# Same 3 evaluation questions
# ============================================================

import json

with open("/content/evaluation_dataset.jsonl", "r", encoding="utf-8") as f:
    evaluation_records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

test_records = [
    r for r in evaluation_records
    if r.get("split") == "test"
]

print("=== BASE GEMMA 3 4B vs KASAI SFT v0.1 ===\n")

for i, record in enumerate(test_records, 1):

    question = record["question"]

    print("=" * 100)
    print(f"TEST {i}: {record['id']}")
    print("QUESTION:")
    print(question)
    print()

    # -----------------------------
    # BASE MODEL
    # -----------------------------
    with model.disable_adapter():
        base_answer = kasai_generate(
            question,
            max_new_tokens=180
        )

    print("BASE GEMMA 3 4B:")
    print(base_answer)
    print()

    # -----------------------------
    # KASAI SFT
    # -----------------------------
    kasai_answer = kasai_generate(
        question,
        max_new_tokens=180
    )

    print("KASAI SFT v0.1:")
    print(kasai_answer)
    print()

=== BASE GEMMA 3 4B vs KASAI SFT v0.1 ===

TEST 1: eval_e01_001
QUESTION:
What is the primary objective of the Nigeria Tax Act 2025?

BASE GEMMA 3 4B:
The primary objective of the Nigeria Tax Act 2024 (often referred to as the Petroleum Profit Tax Act) is to **boost oil and gas production and investment in Nigeria's oil sector, primarily through a revised profit tax regime.**

Here's a breakdown of the key objectives and how it achieves them:

* **Increased Oil Production:** The core aim is to incentivize increased oil production by reducing the tax burden for oil companies. This is done by shifting from a royalty-based system to a profit-based taxation model.
* **Attracting Foreign Direct Investment (FDI):**  The Act is designed to make Nigeria more attractive to international oil companies (IOCs) and encourage them to invest more in Nigerian oil assets. Lower taxes are expected to improve profitability and return on investment.
* **Revenue Maximization for the Federal Government:** W

In [14]:
# ============================================================
# KASAI v0.1 — FULL 12-QUESTION PILOT EVALUATION
# ============================================================

import json
from pathlib import Path

eval_path = "/content/evaluation_dataset.jsonl"

with open(eval_path, "r", encoding="utf-8") as f:
    records = [json.loads(line) for line in f if line.strip()]

print("=== KASAI v0.1 FULL PILOT EVALUATION ===")
print("Questions:", len(records))
print()

full_results = []

for i, record in enumerate(records, 1):

    answer = kasai_generate(
        record["question"],
        max_new_tokens=180
    )

    result = {
        "id": record["id"],
        "category": record["category"],
        "question": record["question"],
        "expected_answer": record["expected_answer"],
        "expected_citations": record.get("expected_citations", []),
        "model_answer": answer,
        "difficulty": record.get("difficulty"),
        "is_repealed_trap": record.get("is_repealed_trap", False),
        "is_unanswerable": record.get("is_unanswerable", False)
    }

    full_results.append(result)

    print("=" * 90)
    print(f"{i}/{len(records)}  {record['id']}  |  {record['category']}")
    print("Q:", record["question"])
    print("A:", answer)

results_path = "/content/kasai_v0.1_full_evaluation.json"

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(full_results, f, ensure_ascii=False, indent=2)

print("\n" + "=" * 90)
print("=== FULL PILOT EVALUATION COMPLETE ===")
print("Saved:", results_path)

=== KASAI v0.1 FULL PILOT EVALUATION ===
Questions: 12

1/12  eval_e06_001  |  E06_Current_Law_Recognition
Q: Which federal revenue agency is charged with executive administration and collection of federal taxes under the 2025 tax reform acts?
A: The Nigeria Revenue Service (NRS) is charged with this function under section 13 of the 2025 Tax Reform Act.

Under section 13 of the 2025 Tax Reform Act, the Nigeria Revenue Service is charged with executive administration and collection of federal taxes.


I'm sorry, but I cannot answer this question from the information I have access to. The relevant section of the 2025 Tax Reform Act is not included in my current knowledge base.

I apologize for the error. I am still under development and learning to access and process all available information. I will strive to improve my knowledge base and provide accurate responses in the future.

I'm unable to answer this question based on my current knowledge.
I don'
2/12  eval_e04_001  |  E04_Citatio

In [15]:
# ============================================================
# KASAI v0.1 — FINAL SUBMISSION PACKAGE
# ============================================================

import os
import json
import shutil
from pathlib import Path
from datetime import datetime

BASE = Path("/content")
MODEL_DIR = BASE / "kasai_sft_v0.1"
PACKAGE_DIR = BASE / "Kasai_v0.1_submission"

# Clean previous package if it exists
if PACKAGE_DIR.exists():
    shutil.rmtree(PACKAGE_DIR)

PACKAGE_DIR.mkdir(parents=True)

# ------------------------------------------------------------
# 1. Copy trained adapter
# ------------------------------------------------------------

adapter_dest = PACKAGE_DIR / "model_adapter"
adapter_dest.mkdir()

for item in MODEL_DIR.iterdir():
    if item.name == "checkpoint-36":
        continue

    if item.is_file():
        shutil.copy2(item, adapter_dest / item.name)

# ------------------------------------------------------------
# 2. Copy evaluation artifacts
# ------------------------------------------------------------

artifacts = [
    "evaluation_dataset.jsonl",
    "kasai_sft_v0.1_test_results.json",
    "kasai_v0.1_full_evaluation.json",
]

for filename in artifacts:
    source = BASE / filename

    if source.exists():
        shutil.copy2(source, PACKAGE_DIR / filename)
        print("Copied:", filename)
    else:
        print("WARNING — missing:", filename)

# ------------------------------------------------------------
# 3. Create submission README
# ------------------------------------------------------------

readme = """# Kasai SLM v0.1

## Project

Kasai is a domain-adapted small language model prototype focused on
Nigerian tax law and tax-administration questions.

## Base Model

google/gemma-3-4b-it

## Adaptation

Supervised Fine-Tuning (SFT) using a LoRA adapter.

Training dataset:
- 96 records
- Nigerian tax-law domain
- Factual accuracy
- Legal interpretation
- Legal reasoning
- Citation
- Current-law recognition
- Repealed-law detection
- Temporal reasoning
- Scenario/application
- Abstention/uncertainty

## Training Configuration

- Epochs: 3
- Batch size: 1
- Gradient accumulation: 8
- Learning rate: 2e-4
- Optimizer: paged_adamw_8bit
- LoRA trainable parameters: approximately 29.8M
- Base model parameters: approximately 2.52B

## Evaluation

A 12-question pilot evaluation dataset was used.

The benchmark covers:
- Current-law recognition
- Factual accuracy
- Citation accuracy
- Citation completeness
- Repealed-law detection
- Temporal reasoning
- Abstention
- Legal interpretation
- Legal reasoning
- Scenario/application

A three-question subset was also used for direct Base Gemma 3 4B
versus Kasai SFT v0.1 comparison.

## Key Findings

Kasai SFT v0.1 demonstrates improved Nigerian tax-domain alignment
compared with the base model on the pilot benchmark.

For example, the base model incorrectly interpreted a question
about the Nigeria Tax Act 2025 as referring to petroleum taxation,
while Kasai produced a response aligned with the 2025 Nigerian tax
reform context.

However, the pilot also identified important limitations:

1. Citation accuracy remains unreliable.
2. The model can produce confident but incorrect statutory provisions.
3. Repealed-law detection requires further improvement.
4. Legal scenario reasoning remains unreliable.
5. The model does not consistently abstain when insufficient facts
   are provided.
6. Some generations terminate incompletely or repeat assistant-style
   content.

## Important Limitation

This is a research prototype and must NOT be treated as authoritative
legal advice.

The evaluation dataset is a small pilot benchmark and is not large
enough to support statistically robust claims of legal accuracy.

For production use, Kasai should be combined with authoritative,
version-controlled Nigerian tax-law retrieval (RAG), citation
verification, temporal/repeal checks, and human legal review.

## Files

model_adapter/
    LoRA adapter and tokenizer files

evaluation_dataset.jsonl
    Pilot evaluation benchmark

kasai_sft_v0.1_test_results.json
    Three-question Base/SFT test results

kasai_v0.1_full_evaluation.json
    Full 12-question pilot evaluation

## Version

Kasai SLM v0.1
"""

with open(PACKAGE_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)

# ------------------------------------------------------------
# 4. Create metadata
# ------------------------------------------------------------

metadata = {
    "project": "Kasai",
    "version": "v0.1",
    "base_model": "google/gemma-3-4b-it",
    "method": "LoRA SFT",
    "training_records": 96,
    "epochs": 3,
    "learning_rate": 2e-4,
    "batch_size": 1,
    "gradient_accumulation_steps": 8,
    "trainable_parameters": 29802496,
    "evaluation_records": 12,
    "evaluation_type": "pilot",
    "created": datetime.now().isoformat(),
    "warning": "Research prototype; not authoritative legal advice."
}

with open(PACKAGE_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

# ------------------------------------------------------------
# 5. Create ZIP
# ------------------------------------------------------------

zip_path = shutil.make_archive(
    str(BASE / "Kasai_v0.1_submission"),
    "zip",
    root_dir=BASE,
    base_dir="Kasai_v0.1_submission"
)

print("\n" + "=" * 70)
print("=== KASAI v0.1 SUBMISSION PACKAGE CREATED ===")
print("=" * 70)

print("Folder:", PACKAGE_DIR)
print("ZIP:", zip_path)

print("\nPackage contents:")
for root, dirs, files in os.walk(PACKAGE_DIR):
    for filename in files:
        relative = Path(root, filename).relative_to(PACKAGE_DIR)
        print(" -", relative)

print("\n✅ KASAI v0.1 IS READY FOR SUBMISSION")

Copied: evaluation_dataset.jsonl
Copied: kasai_sft_v0.1_test_results.json
Copied: kasai_v0.1_full_evaluation.json

=== KASAI v0.1 SUBMISSION PACKAGE CREATED ===
Folder: /content/Kasai_v0.1_submission
ZIP: /content/Kasai_v0.1_submission.zip

Package contents:
 - kasai_sft_v0.1_test_results.json
 - metadata.json
 - README.md
 - kasai_v0.1_full_evaluation.json
 - evaluation_dataset.jsonl
 - model_adapter/tokenizer.json
 - model_adapter/chat_template.jinja
 - model_adapter/README.md
 - model_adapter/tokenizer_config.json
 - model_adapter/adapter_model.safetensors
 - model_adapter/adapter_config.json

✅ KASAI v0.1 IS READY FOR SUBMISSION
